In [ ]:
# install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn
!pip install torch transformers datasets evaluate

In [ ]:
# load dataset
import pandas as pd

df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")
print(df.head())

In [ ]:
# data cleaning （to remove noise and make vocab smaller)
import re #regex library

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)   # remove html
    text = re.sub(r'[^a-zA-Z ]', '', text) # remove chars other than aA-zZ and space
    return text

df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
print(df["clean_review"].head())

In [ ]:
# train test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"],  # input for x
    df["sentiment"],  # target for y
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# use tfidf to vectorize word to number, with importance of each word
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
# train classifier
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_tfidf, y_train) # train the model using x_tfidf (vectorized)

In [ ]:
# test the model
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# try upgrading to BERT
# MAP label into numbers
label_map = {"negative": 0, "positive": 1}

y_train = y_train.map(label_map)
y_test = y_test.map(label_map)

In [ ]:
# create dataframe
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})

In [ ]:
# convert to huggingface dataset, huggingface datasets are optimized for nlp pipeline
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
# load bert tokenizer (tokenize sentences into individual words and finally convert to numbers)
from transformers import AutoTokenizer

# bert base uncased = convert all to lowercase
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# define tokenization function
def tokenize_function(example):
    return tokenizer( # convert text into input id and attention mask
        example["text"], # get the text column value
        padding="max_length", # token max length. if a sentence<256 = PADDING, else = cut because NN need same size input
        truncation=True, # to cut sentence if exceeeds 256
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True) # batched true means process many sentences altogether to make it faster
test_dataset = test_dataset.map(tokenize_function, batched=True)

# before and after BERT tokenization
# before :
# {
#   "text": "I love NLP",
#   "label": 1
# }

# after
# {
#   "text": "I love NLP",
#   "label": 1,
#   "input_ids": [...256 numbers...],
#   "attention_mask": [...256 numbers...]
# }

In [ ]:
# set format for pytorch
# this will convert dataset in PyTorch format, and only keep the columns needed for training
# BERT model works well with torch format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# load BERT model for classification
from transformers import AutoModelForSequenceClassification
#  AutoModel.... is a helper class that loads a model designed to classify text (like positive vs negative, spam vs not spam, etc.).
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Load the pre-trained BERT model that ignores upper/lowercase
    num_labels=2 # tell the model to classify text into 2 categories
)

In [ ]:
# define training args (training rules/settings for model)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # eval after each epoch
    save_strategy="epoch", # save after each epoch
    learning_rate=2e-5, # learning rate 0.00002, small rate avoid breaking the model
    per_device_train_batch_size=16, # number of training samples processed at once
    per_device_eval_batch_size=16, # number of samples processed at once during eval
    num_train_epochs=2,
    weight_decay=0.01, # prevent overfitting by penalizing very large weight
    logging_dir="./logs",
)

In [ ]:
# define Trainer that will execute the training settings
# Trainer will do the model training process for us
from transformers import Trainer
import evaluate  # library to handle all kind of evaluation
import numpy as np # library to deal with numbers and arrays

metric = evaluate.load("accuracy") # use accuracy as the evaluation metric

# below function will be used when the model is evaluated
# logits are raw scores prediction from the BERT model
# labels = correct answers
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1) # convert raw scores to predictions using argmax
    return metric.compute(predictions=predictions, references=labels) # compute accuracy

trainer = Trainer( # set up the trainer
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# begin the training process using Trainer (1h04m of processing time)
trainer.train()

In [ ]:
# evaluate performance
trainer.evaluate()